In [1]:
import pandas as pd

df = pd.read_csv("data/data.txt", sep="|", engine="python")
missing = df.isnull().sum().sort_values(ascending=False)
missing_percent = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    "MissingCount": missing,
    "MissingPercent": missing_percent
})

missing_df[missing_df["MissingCount"] > 0]


,MissingCount,MissingPercent
NumberOfVehiclesInFleet,1000098,100.000000
CrossBorder,999400,99.930207
CustomValueEstimate,779642,77.956560
Rebuilt,641901,64.183810
Converted,641901,64.183810
WrittenOff,641901,64.183810
NewVehicle,153295,15.327998
Bank,145961,14.594670
AccountType,40232,4.022806
Gender,9536,0.953507


In [2]:
df.drop(columns=[
    "NumberOfVehiclesInFleet",
    "CrossBorder",
    "Rebuilt",
    "Converted",
    "WrittenOff"
], inplace=True)
df["CustomValueEstimate"] = df["CustomValueEstimate"].fillna(
    df["CustomValueEstimate"].median()
)
df["NewVehicle"] = df["NewVehicle"].fillna("Unknown")
df["Bank"] = df["Bank"].fillna("Unknown")
df["AccountType"] = df["AccountType"].fillna("Unknown")
df["Gender"] = df["Gender"].fillna(df["Gender"].mode()[0])
df["MaritalStatus"] = df["MaritalStatus"].fillna("Unknown")
num_cols = [
    "Cylinders",
    "kilowatts",
    "NumberOfDoors",
    "cubiccapacity"
]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
    cat_cols = ["VehicleType", "make", "Model", "bodytype"]




Missing values were handled using a structured approach based on the proportion of missingness and variable importance. Features with excessive missingness (>60%) were removed as they provided limited analytical value. Key predictive variables such as vehicle characteristics and insured value were retained and imputed using median values for numerical features and “Unknown” for categorical variables. This approach preserves data integrity while minimizing bias introduced by missing data.

In [3]:
df["VehicleAge"] = 2026 - df["RegistrationYear"]
df["TransactionMonth"] = pd.to_datetime(df["TransactionMonth"], errors="coerce")
df["PolicyYear"] = df["TransactionMonth"].dt.year
df["PolicyMonth"] = df["TransactionMonth"].dt.month
df["PowerToValue"] = df["kilowatts"] / df["CustomValueEstimate"]
df["EnginePowerRatio"] = df["kilowatts"] / df["cubiccapacity"]
df["PremiumToValueRatio"] = df["TotalPremium"] / df["SumInsured"]
df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)
df["LossRatio"] = df["TotalClaims"] / df["TotalPremium"]
df.replace([float("inf"), -float("inf")], None, inplace=True)
num_cols = df.select_dtypes(include=["number"]).columns
cat_cols = df.select_dtypes(include=["object", "string"]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna("Unknown")


Missing values were handled by applying type-specific imputation. Numerical features were filled using median values to reduce sensitivity to outliers, while categorical features were assigned an "Unknown" category to preserve information without introducing bias.

In [4]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns
cat_cols
df = pd.get_dummies(df, columns=[
    "Gender",
    "VehicleType",
    "MaritalStatus",
    "Bank",
    "AccountType"
], drop_first=True)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["Model"] = le.fit_transform(df["Model"].astype(str))
df["bodytype"] = le.fit_transform(df["bodytype"].astype(str))
df.select_dtypes(include=["object", "string"]).columns
df = pd.get_dummies(df, drop_first=True)

Categorical variables were encoded using a combination of one-hot encoding and label encoding. One-hot encoding was applied to nominal variables with low cardinality to avoid introducing ordinal bias, while label encoding was used for high-cardinality features such as vehicle models to reduce dimensionality. This ensures compatibility with machine learning algorithms while preserving interpretability.

In [9]:
df_severity = df[df["TotalClaims"] > 0].copy()

X = df_severity.drop(columns=["TotalClaims"])
X = X.drop(columns=["TransactionMonth", "VehicleIntroDate"], errors="ignore")
X = pd.get_dummies(X, drop_first=True)
y = df_severity["TotalClaims"]
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X = df.drop(columns=["HasClaim"])
y = df["HasClaim"]
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [10]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("LR RMSE:", rmse_lr)
print("LR R2:", r2_lr)

LR RMSE: 35631.91644517475
LR R2: 0.21054869766476303


The Linear Regression model achieved an R² of 0.21, indicating limited ability to explain variation in claim severity. This suggests that claim amounts are influenced by non-linear relationships and complex interactions between variables, which are not well captured by linear assumptions. The relatively high RMSE further confirms that the model has large prediction errors and is not suitable as a standalone pricing model.

In [13]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest RMSE:", rmse_rf)
print("Random Forest R2:", r2_rf)

Random Forest RMSE: 36049.76585447038
Random Forest R2: 0.19192461182273435


The Random Forest regression model was applied to predict claim severity (TotalClaims) using the engineered features. The model achieved an RMSE of 36,049.77, indicating a relatively high average prediction error in monetary terms. The R² score of 0.192 shows that the model explains approximately 19.2% of the variance in claim amounts.

These results suggest that while Random Forest is capable of capturing non-linear relationships in the data, the overall predictive performance remains limited. This is likely due to the high variability and skewed nature of insurance claim amounts, as well as the presence of extreme outliers in the dataset. Further improvements such as target transformation, advanced feature engineering, or more powerful ensemble methods may be required to enhance model performance for production-level risk-based pricing.

35645.73350521122 0.20993632471280954


In [14]:
from xgboost import XGBRegressor
xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print("XGBoost RMSE:", rmse_xgb)
print("XGBoost R2:", r2_xgb)

XGBoost RMSE: 35955.75100668895
XGBoost R2: 0.19613390526041574


Model	                RMSE	R²
Linear Regression  35631.92  	0.2105
Random Forest	   36049.77	    0.1919
XGBoost	           35955.75 	0.1961

Three machine learning models (Linear Regression, Random Forest, and XGBoost) were evaluated for predicting claim severity. All models achieved similar performance, with R² values around 0.19–0.21, indicating limited explanatory power. Linear Regression performed marginally better, suggesting that the current feature set does not contain strong non-linear signals that tree-based models can exploit. The results indicate that claim severity is influenced by factors not captured in the dataset, such as driver behavior, environmental conditions, or historical claim patterns. This highlights the need for additional feature engineering and external data sources to improve predictive accuracy in a production risk-pricing system.